# DINOv3 Adaptive Zoom — final demo

Eğitilmiş checkpoint'i yükler; eğitim veya 100/1399 görüntülük değerlendirme başlatmaz.
Yeni Colab oturumunda T4 GPU seç; 1–6 hücrelerini sırayla çalıştır.
Model/veri zaten hazırsa yalnızca **5 ve 6** yeterlidir.
Mevcut deney protokolü: 224×224 input, entropy, configurable Top-K,
original-image crops, aynı modelle second pass, overlap'te local logits ortalaması.
Varsayılan K=3; K=1 veya K=5 de seçilebilir. NMS nedeniyle gerçek seçim sayısı K'dan az olabilir.
Özel fotoğrafta GT yoksa doğruluk metriği hesaplanmaz.

Doğruluk kıyaslaması: 1399 görüntüde baseline 0.7133, K=3 0.7220, sliding 0.7325.
Son ayrı 100 görüntülük T4 benchmark'ında K=3 147.5 ms, sliding 510.3 ms.
Bu tek görüntü demosu genel değerlendirmeyi veya bilimsel benchmark'ı tekrarlamaz.


## 1 — Bağımlılıklar
Colab yeniden başlatma isterse yeniden başlatıp 1'den devam et.

In [ ]:
%pip install -q "transformers==5.17.0" "huggingface_hub==1.33.0"


## 2 — Checkpoint ve sonuç paketi
Tek bir tam deney ZIP'i seç. Galeri ZIP'i yerine checkpoint içeren paket kullan.

In [ ]:
import json
import math
import random
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import files

assert torch.cuda.is_available(), "Çalışma zamanı türünü T4 GPU olarak seçin."
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
session_dir = Path("/content/DINOv3_sessions") / session_id
session_dir.mkdir(parents=True, exist_ok=False)

uploaded = files.upload()
assert len(uploaded) == 1, "Tek bir ZIP veya PTH yükleyin."
upload_name, upload_data = next(iter(uploaded.items()))
upload_path = session_dir / Path(upload_name).name
upload_path.write_bytes(upload_data)

if upload_path.suffix.lower() == ".zip":
    extract_dir = session_dir / "recovered"
    extract_dir.mkdir()
    with zipfile.ZipFile(upload_path) as archive:
        for member in archive.infolist():
            destination = (extract_dir / member.filename).resolve()
            if not destination.is_relative_to(extract_dir.resolve()):
                raise ValueError("ZIP içinde geçersiz dosya yolu.")
        archive.extractall(extract_dir)
    candidates = list(extract_dir.rglob("head_epoch_05.pth"))
    assert len(candidates) == 1, "ZIP içinde tek bir head_epoch_05.pth olmalı."
    checkpoint_path = candidates[0]
elif upload_path.suffix.lower() == ".pth":
    checkpoint_path = upload_path
else:
    raise ValueError("ZIP veya PTH dosyası yükleyin.")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert "head_state_dict" in checkpoint and "config" in checkpoint
config = checkpoint["config"]
model_name = config["model_name"]
revision = config["revision"]
assert model_name == "facebook/dinov3-vits16-pretrain-lvd1689m"
assert config["input_size"] == 224 and config["num_classes"] == 21
run_dir = checkpoint_path.parent
(run_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", checkpoint_path)
print("Kaydedilen epoch:", checkpoint["epoch"])
print("Çalışma klasörü:", run_dir)


## 3 — Backbone ve eğitilmiş head
Hugging Face hesabının DINOv3 erişimi gerekir. Token'ı paylaşma.

In [ ]:
import transformers
import torchvision
from huggingface_hub import notebook_login
from transformers import AutoImageProcessor, AutoModel

notebook_login()
processor = AutoImageProcessor.from_pretrained(model_name, revision=revision)
model = AutoModel.from_pretrained(model_name, revision=revision).to(device)
model.requires_grad_(False)
model.eval()

class DinoSegmentationModel(nn.Module):
    def __init__(self, backbone, num_classes=21):
        super().__init__()
        self.backbone = backbone
        self.backbone.requires_grad_(False)
        self.backbone.eval()
        self.segmentation_head = nn.Conv2d(backbone.config.hidden_size, num_classes, 1)

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, pixel_values):
        batch, _, height, width = pixel_values.shape
        patch_size = self.backbone.config.patch_size
        with torch.no_grad():
            hidden = self.backbone(pixel_values=pixel_values).last_hidden_state
        patches = hidden[:, 1 + self.backbone.config.num_register_tokens:, :]
        gh, gw = height // patch_size, width // patch_size
        assert patches.shape[1] == gh * gw
        features = patches.transpose(1, 2).reshape(batch, -1, gh, gw)
        logits = self.segmentation_head(features.to(self.segmentation_head.weight.dtype))
        return F.interpolate(logits, size=(height, width), mode="bilinear", align_corners=False)

restored_model = DinoSegmentationModel(model).to(device)
restored_model.segmentation_head.load_state_dict(checkpoint["head_state_dict"], strict=True)
restored_model.eval()
seg_model = restored_model
assert all(not p.requires_grad for p in model.parameters())
assert (model.config.hidden_size, model.config.patch_size,
        model.config.num_register_tokens) == (384, 16, 4)
for key, value in restored_model.segmentation_head.state_dict().items():
    assert torch.equal(value.cpu(), checkpoint["head_state_dict"][key])
environment = {"torch": str(torch.__version__), "torchvision": torchvision.__version__,
               "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0)}
(run_dir / f"environment_{session_id}.json").write_text(json.dumps(environment, indent=2))
print("Backbone ve eğitilmiş head yüklendi. Yeniden eğitim yapılmadı.")
print(environment)


## 4 — VOC validation
Veri varsa yeniden indirmez. Özel fotoğraf demosunda bu hücre atlanabilir.

In [ ]:
from torchvision.datasets import VOCSegmentation
voc_root = Path("/content/VOC")
voc_ready = (voc_root / "VOCdevkit/VOC2012/ImageSets/Segmentation/val.txt").is_file()
raw_val_dataset = VOCSegmentation(str(voc_root), year="2012", image_set="val",
                                 download=not voc_ready)
assert len(raw_val_dataset) == 1449
print("Validation images:", len(raw_val_dataset))


## 5 — Selector, second pass, fusion ve görselleştirme
GT selector'a verilmez; yalnızca sonuç metrikleri için kullanılır. Bu hücre inference başlatmaz.

In [ ]:
"""Colab demo helpers. Execute after loading restored_model and processor."""
import json
import math
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw, ImageOps
from torchvision.transforms import InterpolationMode


def prepare_input(image):
    resized = TF.resize(image.convert("RGB"), [224, 224],
                        interpolation=InterpolationMode.BICUBIC)
    tensor = TF.normalize(TF.to_tensor(resized),
                          processor.image_mean, processor.image_std)
    return tensor.unsqueeze(0).to(device)


@torch.inference_mode()
def predict_logits(image):
    restored_model.eval()
    return restored_model(prepare_input(image)).float().cpu()


def normalized_entropy(logits):
    probabilities = logits.float().softmax(1)
    return (-(probabilities * probabilities.clamp_min(1e-12).log()).sum(1)
            / math.log(logits.shape[1])).clamp(0, 1)


def confusion_for(prediction, target):
    prediction = torch.as_tensor(prediction, dtype=torch.int64).cpu()
    target = torch.as_tensor(target, dtype=torch.int64).cpu()
    assert prediction.shape == target.shape
    valid = target != 255
    gt, pred = target[valid], prediction[valid]
    assert torch.all((gt >= 0) & (gt < 21))
    assert torch.all((pred >= 0) & (pred < 21))
    return torch.bincount(gt * 21 + pred, minlength=441).reshape(21, 21)


def image_miou(prediction, target):
    matrix = confusion_for(prediction, target).double()
    diagonal = matrix.diag()
    union = matrix.sum(0) + matrix.sum(1) - diagonal
    present = union > 0
    return (diagonal[present] / union[present]).mean().item() if present.any() else None


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    intersection = (max(0, min(ax2, bx2) - max(ax1, bx1))
                    * max(0, min(ay2, by2) - max(ay1, by1)))
    union = ((ax2 - ax1) * (ay2 - ay1)
             + (bx2 - bx1) * (by2 - by1) - intersection)
    return intersection / union if union > 0 else 0.0


def select_entropy_topk(entropy_map, k=3, crop_size=112, stride=28,
                        overlap_threshold=0.30):
    entropy_map = torch.as_tensor(entropy_map).detach().float().cpu()
    assert entropy_map.ndim == 2 and torch.isfinite(entropy_map).all()
    assert isinstance(k, int) and k > 0
    assert isinstance(crop_size, int) and crop_size > 0
    assert isinstance(stride, int) and stride > 0
    assert 0 <= overlap_threshold <= 1
    height, width = entropy_map.shape
    assert height > 0 and width > 0
    crop_h, crop_w = min(crop_size, height), min(crop_size, width)

    def positions(length, size):
        starts = list(range(0, length - size + 1, stride))
        if starts[-1] != length - size:
            starts.append(length - size)
        return starts

    candidates = []
    for y in positions(height, crop_h):
        for x in positions(width, crop_w):
            candidates.append({
                "box": [x, y, x + crop_w, y + crop_h],
                "score": entropy_map[y:y + crop_h, x:x + crop_w].mean().item(),
            })
    ranked = sorted(candidates, key=lambda region: region["score"], reverse=True)
    selected = []
    for candidate in ranked:
        if all(box_iou(candidate["box"], previous["box"]) <= overlap_threshold
               for previous in selected):
            selected.append({"rank": len(selected) + 1,
                             "box": candidate["box"].copy(),
                             "score": candidate["score"]})
            if len(selected) == k:
                break
    return selected, ranked


def topk_box_to_original(box, image_size, map_size=(224, 224)):
    width, height = image_size
    map_h, map_w = map_size
    x1, y1, x2, y2 = box
    assert 0 <= x1 < x2 <= map_w and 0 <= y1 < y2 <= map_h
    result = [max(0, math.floor(x1 * width / map_w)),
              max(0, math.floor(y1 * height / map_h)),
              min(width, math.ceil(x2 * width / map_w)),
              min(height, math.ceil(y2 * height / map_h))]
    assert result[0] < result[2] and result[1] < result[3]
    return result


def run_adaptive_zoom(image, k=3, crop_size=112, stride=28,
                      overlap_threshold=0.30):
    """No GT is accepted: selection and fusion depend only on the image."""
    image = image.convert("RGB")
    width, height = image.size
    low = predict_logits(image)
    global_logits = F.interpolate(low, size=(height, width),
                                  mode="bilinear", align_corners=False)
    entropy = normalized_entropy(low)[0]
    regions, candidates = select_entropy_topk(
        entropy, k, crop_size, stride, overlap_threshold)
    local_sum = torch.zeros_like(global_logits)
    coverage = torch.zeros((height, width), dtype=torch.int32)
    local_outputs = []
    for region in regions:
        box = topk_box_to_original(region["box"], image.size, tuple(entropy.shape))
        region["original_box"] = box
        x1, y1, x2, y2 = box
        crop = image.crop(tuple(box))
        crop_low = predict_logits(crop)
        local = F.interpolate(crop_low, size=(y2 - y1, x2 - x1),
                              mode="bilinear", align_corners=False)
        assert torch.isfinite(local).all()
        local_sum[:, :, y1:y2, x1:x2] += local
        coverage[y1:y2, x1:x2] += 1
        local_outputs.append({"crop": crop, "logits_224": crop_low,
                              "prediction": local.argmax(1)[0]})
    covered = coverage > 0
    average = local_sum / coverage.clamp_min(1)[None, None]
    fused = torch.where(covered[None, None], average, global_logits)
    assert torch.isfinite(fused).all()
    assert torch.equal(fused[0, :, ~covered], global_logits[0, :, ~covered])
    return {"baseline": global_logits.argmax(1)[0],
            "final": fused.argmax(1)[0], "global_logits_224": low,
            "entropy": entropy, "regions": regions, "candidates": candidates,
            "locals": local_outputs, "coverage": coverage,
            "parameters": {"k_requested": k, "k_selected": len(regions),
                           "crop_size": crop_size, "stride": stride,
                           "overlap_threshold": overlap_threshold, "padding": 0}}


def voc_palette():
    palette = np.zeros((256, 3), dtype=np.uint8)
    for index in range(256):
        value = index
        for bit in range(8):
            for channel in range(3):
                palette[index, channel] |= ((value >> channel) & 1) << (7 - bit)
            value >>= 3
    palette[255] = [160, 160, 160]
    return palette


def colorize(mask):
    array = mask.detach().cpu().numpy() if torch.is_tensor(mask) else np.asarray(mask)
    return voc_palette()[array.astype(np.uint8)]


def save_and_show_demo(image, result, output_dir, target=None, source_name="image"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    image.save(output_dir / "original.png")
    if target is not None:
        assert tuple(target.shape) == (image.height, image.width)
        Image.fromarray(target.numpy().astype(np.uint8)).save(output_dir / "ground_truth.png")

    for name in ("baseline", "final"):
        labels = result[name].numpy().astype(np.uint8)
        Image.fromarray(labels).save(output_dir / f"{name}_labels.png")
        Image.fromarray(colorize(labels)).save(output_dir / f"{name}_color.png")

    np.savez_compressed(output_dir / "global_output.npz",
                        logits_224=result["global_logits_224"].numpy(),
                        entropy=result["entropy"].numpy(),
                        coverage=result["coverage"].numpy())

    tiles = []
    for region, local in zip(result["regions"], result["locals"]):
        rank = region["rank"]
        local["crop"].save(output_dir / f"crop_{rank:02d}.png")
        np.savez_compressed(output_dir / f"crop_{rank:02d}_output.npz",
                            logits_224=local["logits_224"].numpy(),
                            prediction_native=local["prediction"].numpy().astype(np.uint8),
                            original_box=np.array(region["original_box"]))
        tile = Image.new("RGB", (224, 248), "white")
        colored = Image.fromarray(colorize(local["prediction"]))
        thumbnail = ImageOps.contain(colored, (224, 224), Image.Resampling.NEAREST)
        tile.paste(thumbnail, ((224 - thumbnail.width) // 2, 24))
        ImageDraw.Draw(tile).text((6, 5), f"Crop {rank}", fill="black")
        tiles.append(tile)

    columns = min(3, len(tiles))
    rows = math.ceil(len(tiles) / columns)
    montage = Image.new("RGB", (224 * columns, 248 * rows), "white")
    for i, tile in enumerate(tiles):
        montage.paste(tile, ((i % columns) * 224, (i // columns) * 248))
    montage.save(output_dir / "local_predictions.png")

    metrics = {}
    for name in ("baseline", "final"):
        if target is not None:
            valid = target != 255
            matrix = confusion_for(result[name], target)
            metrics[name] = {
                "image_miou": image_miou(result[name], target),
                "pixel_accuracy": (float(matrix.diag().sum() / matrix.sum())
                                   if matrix.sum() > 0 else None),
                "wrong_pixels": int(((result[name] != target) & valid).sum()),
                "confusion": matrix.tolist(),
            }

    # Seven panels; local predictions are shown as a montage.
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    axes[0].imshow(image); axes[0].set_title("Original")
    if target is not None:
        axes[1].imshow(colorize(target)); axes[1].set_title("GT — gray: ignore")
    else:
        axes[1].text(0.5, 0.5, "Ground truth unavailable", ha="center", va="center")
        axes[1].set_title("Ground truth")
    axes[2].imshow(colorize(result["baseline"])); axes[2].set_title("Global prediction")
    im = axes[3].imshow(result["entropy"].numpy(), cmap="inferno", vmin=0, vmax=1)
    axes[3].set_title("Normalized entropy")
    fig.colorbar(im, ax=axes[3], fraction=0.046)
    axes[4].imshow(image); axes[4].set_title(f"Top-K crops — {len(tiles)} selected")
    for region in result["regions"]:
        x1, y1, x2, y2 = region["original_box"]
        color = plt.get_cmap("tab10")((region["rank"] - 1) % 10)
        axes[4].add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                    fill=False, edgecolor=color, linewidth=2))
        axes[4].text(x1, y1, str(region["rank"]), color=color, backgroundcolor="black")
    axes[5].imshow(montage); axes[5].set_title("Local predictions")
    axes[6].imshow(colorize(result["final"])); axes[6].set_title("Final segmentation")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(source_name)
    fig.tight_layout()
    fig.savefig(output_dir / "demo_7_panels.png", dpi=150)
    plt.show()
    plt.close(fig)

    report = {"source": source_name, "image_size": list(image.size),
              "parameters": result["parameters"],
              "regions": result["regions"], "candidates": result["candidates"],
              "passes": {"baseline": 1, "adaptive": 1 + len(tiles)},
              "union_area_percent": float((result["coverage"] > 0).float().mean() * 100),
              "metrics": metrics,
              "metric_note": "Native resolution, ignore=255; no metrics without GT.",
              "fusion": "Mean local logits inside crop union; global outside.",
              "checkpoint": str(checkpoint_path),
              "model_name": model_name, "revision": revision}
    (output_dir / "results.json").write_text(
        json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    if metrics:
        print("Baseline image mIoU:", metrics["baseline"]["image_miou"])
        print("Adaptive image mIoU:", metrics["final"]["image_miou"])
    else:
        print("GT yok; doğruluk metrikleri hesaplanmadı.")
    print("Requested / selected K:", result["parameters"]["k_requested"], "/", len(tiles))
    print("Backbone passes:", 1 + len(tiles))
    print("Saved:", output_dir)
    return report


## 6 — Tek görüntülük demo
Parametreleri ayarla, çalıştır. Yedi panel ve yerel logits/koordinatlar ZIP olarak indirilir.
Panel sırası: Original, GT, global prediction, entropy, crop kutuları, local predictions, final segmentation.

In [ ]:
# Bu hücrede parametreleri değiştirip yeniden çalıştırabilirsin.
SOURCE = "voc"          # "voc" veya "upload"
IMAGE_INDEX = 9
K = 3
CROP_SIZE = 112
STRIDE = 28
OVERLAP_THRESHOLD = 0.30

assert SOURCE in ("voc", "upload")
if SOURCE == "voc":
    assert "raw_val_dataset" in globals(), "Önce VOC veri hücresini çalıştır."
    assert 0 <= IMAGE_INDEX < len(raw_val_dataset)
    demo_image, demo_mask = raw_val_dataset[IMAGE_INDEX]
    demo_image = demo_image.convert("RGB")
    demo_target = torch.from_numpy(np.array(demo_mask, dtype=np.int64, copy=True))
    demo_name = f"VOC index {IMAGE_INDEX} | {Path(raw_val_dataset.images[IMAGE_INDEX]).stem}"
else:
    uploaded_photo = files.upload()
    assert len(uploaded_photo) == 1, "Tek bir fotoğraf seç."
    import io
    photo_name, photo_bytes = next(iter(uploaded_photo.items()))
    with Image.open(io.BytesIO(photo_bytes)) as photo:
        demo_image = ImageOps.exif_transpose(photo).convert("RGB")
    demo_target = None
    demo_name = Path(photo_name).name

demo_result = run_adaptive_zoom(
    demo_image, k=K, crop_size=CROP_SIZE, stride=STRIDE,
    overlap_threshold=OVERLAP_THRESHOLD,
)

# Sabit referans örneğinde önceki deneyle eşleşme kontrolü.
if SOURCE == "voc" and IMAGE_INDEX == 9 and CROP_SIZE == 112 and STRIDE == 28 and OVERLAP_THRESHOLD == 0.30:
    expected = {1: 0.9214, 3: 0.9512, 5: 0.9597}
    assert abs(image_miou(demo_result["baseline"], demo_target) - 0.8301) < 0.0001
    if K in expected:
        assert len(demo_result["regions"]) == K
        assert abs(image_miou(demo_result["final"], demo_target) - expected[K]) < 0.0001
    print("Önceki referans görüntü sonuçlarıyla eşleşti.")

demo_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
demo_dir = Path(run_dir) / f"final_demo_{demo_id}"
demo_report = save_and_show_demo(demo_image, demo_result, demo_dir,
                                 target=demo_target, source_name=demo_name)
demo_zip = shutil.make_archive(str(demo_dir), "zip",
                               root_dir=str(demo_dir.parent), base_dir=demo_dir.name)
print("PC'ye indirilecek demo paketi:", demo_zip)
files.download(demo_zip)


## 7 — Oturum sonunda tam yedek (isteğe bağlı)
Notebook'u ayrıca Dosya → İndir → .ipynb ile kaydet.

In [ ]:
# Oturum sonunda checkpoint ve bütün sonuçları tek paket halinde indir.
# Bu ZIP notebook kodunu içermez; .ipynb dosyasını da ayrıca sakla.
assert Path(run_dir).is_dir()
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
full_zip = shutil.make_archive(f"/content/DINOv3_complete_{timestamp}", "zip",
                              root_dir=str(Path(run_dir).parent), base_dir=Path(run_dir).name)
print("Tam deney paketi:", full_zip)
files.download(full_zip)
